# ViSta: stacking and line measurement in the uv plane — a worked example

This notebook runs the whole of ViSta on a **simulated sample**, where the true
answer is known, so every step can be checked and every option tried:

1. **simulate** a sample of ALMA-like observations of faint galaxies (one
   Measurement Set each, the line too weak to be seen in any of them);
2. **stack** them: every dataset brought to the rest frame and to the
   position of its galaxy, then combined — with the continuum subtraction and
   the compression into statistics done in the same pass;
3. **measure the line** with the joint fit (`measure_line`): profile, flux,
   width, size, significance, noise diagnostics;
4. see **what each option changes**: source model, size prior, noise model,
   weighting, redshift rescaling, primary beam, window;
5. the **continuum**, measured from the same stack;
6. **redo the continuum subtraction** with another window, without stacking
   again;
7. the **older chain** (`extract_flux`: template + uv fit) for comparison;
8. the **figures in your own style** (`examples/plot_line.py`);
9. the **image plane** (optional, needs CASA);
10. a small **Monte Carlo**: is the error bar right?
11. how to run the same on **your data**.

On Colab: *Runtime → Run all*. Everything runs on the CPU in about 10 minutes
(the CASA part adds a few minutes). Every parameter is in the code cells —
change it and run the cell again.

## 0. Setup

The code is cloned from GitHub and the C++/OpenMP kernel (`ms_ops`) is built
with CMake. If the repository is private, add a GitHub token to the Colab
secrets (key icon on the left) with the name `GITHUB_TOKEN`; or upload a zip
of the repository, unzip it in `/content` and the cell will use it.

In [ ]:
REPO_URL = "https://github.com/martitors/ViSta-HPC.git"
BRANCH = "extract_opt"
INSTALL_CASA = False     # True: also casatasks/casatools, for the image-plane section (~2 min)

import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = os.environ.get("VISTA_REPO")            # a local checkout, to run this outside Colab
if REPO is None:
    REPO = "/content/ViSta-HPC" if IN_COLAB else os.path.abspath("ViSta-HPC")
    if not os.path.isdir(REPO):
        url = REPO_URL
        if IN_COLAB:
            try:                                 # private repository: token from the Colab secrets
                from google.colab import userdata
                url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
            except Exception:
                pass
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
print("ViSta source:", REPO)


def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(cmd)


if IN_COLAB:
    sh(f"pip install -q -r {REPO}/requirements.txt pybind11")
    if INSTALL_CASA:
        sh("pip install -q casatasks casatools")
if not glob.glob(f"{REPO}/vista/ms_ops*.so"):          # build the kernel once
    sh(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release > /dev/null"
       f" && cmake --build {REPO}/build -j 2 > /dev/null && cmake --install {REPO}/build > /dev/null")

sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "examples"))   # simulate.py, plot_line.py
import vista
from vista import ms_ops                               # the compiled kernel
print("ViSta", vista.__version__, "· kernel", os.path.basename(ms_ops.__file__))

In [ ]:
import json, time, shutil
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

WORK = "/content/vista_demo" if IN_COLAB else os.path.abspath("vista_demo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


def show_png(paths, width=720):
    for p in paths:
        display(Image(p, width=width))

## 1. A simulated sample

`examples/simulate.py` writes Measurement Sets of a compact ALMA-like array
(43 antennas of 12 m, beam ≈ 1″ at 230 GHz) observing a galaxy at redshift
*z*, a couple of arcsec from the phase centre. The galaxy has

* an emission line (here CO(3–2), rest 345.796 GHz): a Gaussian of integrated
  flux `line_flux`, width `fwhm_kms`;
* a continuum `cont_flux` (with a dust-like spectral index);
* a size: a circular Gaussian of FWHM `size_arcsec` (0 = a point source);
* the primary-beam attenuation at its position;
* Gaussian noise, with weights wrong by `kappa` and channels correlated as
  after Hanning smoothing — the two things that make the noise of real data
  differ from what the weights say.

With `same_luminosity=True` every galaxy has the luminosity that gives
`line_flux` at the median redshift `z_ref`: after the redshift rescaling the
stack must return exactly `line_flux`. **Change the parameters below**: fewer
sources, more noise, a point source (`size_arcsec=0`), a larger size,
redshift errors (`v_jitter_kms`), a spread of luminosities
(`flux_scatter_dex`), no continuum...

In [ ]:
import simulate

REST_GHZ = 345.79599          # CO(3-2)
SIM = dict(
    n_sources=12,             # galaxies in the stack
    z_range=(0.4, 0.6),       # redshifts, uniform in this range
    line_flux=0.25,           # Jy km/s at z_ref: what the stack should give
    fwhm_kms=250.0,           # line FWHM
    size_arcsec=0.8,          # source FWHM (0 = point source); the beam is ~1"
    cont_flux=3e-4,           # continuum at the line frequency, Jy (0 = none)
    noise=0.08,               # Jy per visibility (real or imaginary part) for weight 1
    kappa=1.3,                # the weights misjudge the noise by this factor
    hanning=True,             # adjacent channels correlated (Hanning smoothing)
    v_jitter_kms=0.0,         # redshift errors: random shift of the line centres
    flux_scatter_dex=0.0,     # spread of the luminosities
    depth_spread_dex=0.2,     # datasets of different depth: noise x 10^(+-0.2)
    same_luminosity=True,
    seed=0,
)
t0 = time.time()
truth = simulate.make_sample("sim", rest_freq_ghz=REST_GHZ, **SIM)
LIST = truth["input_list"]
TRUE_F, TRUE_C = truth["line_flux_stack"], truth["cont_flux_stack"]
print(f"{SIM['n_sources']} Measurement Sets in {time.time() - t0:.0f} s, z_ref = {truth['z_ref']:.3f}")
print(f"expected from the stack: line {TRUE_F:.3f} Jy km/s, continuum {TRUE_C * 1e3:.3f} mJy\n")
print(f"{'dataset':18s} {'z':>6s} {'F_line':>8s} {'peak':>9s} {'noise':>6s} {'offset':>7s} {'PB':>6s}")
for s in truth["sources"]:
    print(f"{os.path.basename(s['path']):18s} {s['z']:6.3f} {s['line_flux']:8.3f} "
          f"{s['line_peak'] * 1e3:6.2f} mJy {s['noise']:6.3f} {s['offset_arcsec']:6.2f}\" "
          f"{s['primary_beam']:6.3f}")
print("\nthe input list:\n" + open(LIST).read())

The input list is the plain ViSta one: `<ms_path> <redshift> <RA> <Dec>
[FIELD_ID] [SPW_IDS] [NORM]`, with the position of the galaxy (not of the
phase centre). One galaxy alone, phase-shifted to its position and averaged
over all baselines, is barely visible — and fainter than its true flux (red),
because the long baselines resolve it:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, s in zip(axes, truth["sources"][:3]):
    v, spec, err = simulate.spectrum_at(s["path"], s["ra_deg"], s["dec_deg"], REST_GHZ, s["z"])
    ax.step(v, spec * 1e3, where="mid", color="k", lw=0.8)
    ax.fill_between(v, -err * 1e3, err * 1e3, step="mid", color="0.8")
    ax.plot(v, s["line_peak"] * s["primary_beam"] * 1e3 * np.exp(-0.5 * (v / (s["fwhm_kms"] / 2.355)) ** 2)
            + s["cont_flux"] * s["primary_beam"] * 1e3, color="C3", label="true (zero spacing)")
    ax.set_title(f"{os.path.basename(s['path'])}, z = {s['z']:.3f}", fontsize=9)
    ax.set_xlabel("velocity [km/s]")
axes[0].set_ylabel("mean Re V [mJy]")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Stacking

`ViSta(...).run(...)` reads every dataset, shifts the phase centre to the
galaxy, brings the frequencies to the rest frame and regrids them on a common
velocity grid, and writes the stack (a Zarr store; a Measurement Set if the
name ends in `.ms`).

Given `line`, `continuum` and `stats_line`, it also does, **in the same pass**:

* the **continuum subtraction**, row by row, on the channels outside the line
  window: the fitted continuum goes to `MODEL_DATA`, the line to
  `CORRECTED_DATA`, and `DATA` is kept (so the subtraction can be redone, §6);
* the **compression**: for every dataset, channel and annulus of baseline
  length, the sums the extraction needs (`line_stats.npy` from
  `CORRECTED_DATA`, `cont_stats.npy` from `MODEL_DATA`). From here on nothing
  reads the stack again: the measurement takes seconds and can be repeated
  at will.

| Parameter | Meaning |
|---|---|
| `central_freq` | rest-frame centre of the output grid (the line) |
| `velocity_range_kms` | output channels from −X to +X km/s |
| `channel_width_hz` | default: the coarsest rest-frame channel of the sample |
| `LineConfig.v_window_kms` | where the line is: left out of the continuum fit |
| `ContinuumConfig.order` | 0 constant, 1 slope, −1 chosen by the BIC per dataset |
| `BinningConfig` | the annuli: from `b_min` to `b_max` kλ, `n_bins` log-spaced |
| `frame` | `"LSRK"`/`"BARY"` if the input frequencies are TOPO |
| `n_readers`, `n_threads`, `device` | parallelism (`device="gpu"` with a CUDA build) |

In [ ]:
from vista import ViSta
from vista.extract import (LineConfig, ContinuumConfig, BinningConfig, WeightingConfig,
                           LineFitConfig, BootstrapConfig, ProfileConfig, UVFitConfig,
                           measure_line, extract_flux, subtract_continuum, compress_visibilities)

line = LineConfig(rest_freq_ghz=REST_GHZ,
                  v_window_kms=(-400, 400))       # the line: excluded from the continuum fit
contsub = ContinuumConfig(order=0,               # a constant per row (1: slope, -1: BIC)
                          exclude_kms=400)        # |v| < 400 km/s left out of the fit
binning = BinningConfig(b_min_klambda=3, b_max_klambda=3000, n_bins=18)

t0 = time.time()
ViSta(LIST, chunk_rows=5000, verbose=True).run(
    output="stack.zarr",
    central_freq=line.rest_freq_hz,
    velocity_range_kms=900,
    line=line, continuum=contsub,
    stats_line="line_stats.npy", stats_continuum="cont_stats.npy",
    binning=binning)
print(f"\nstacked in {time.time() - t0:.0f} s")

A first look at the stack: the visibilities averaged over all baselines and
all galaxies, before (`DATA`) and after (`CORRECTED_DATA`) the continuum
subtraction, with the fitted continuum (`MODEL_DATA`). This is *not* the
flux: a resolved source has less signal on the long baselines, and neither
the redshift rescaling nor the primary-beam correction is applied yet. It is
the place to check that the line sits in the window and that the line-free
channels are flat.

In [ ]:
from daskms.experimental.zarr import xds_from_zarr

datasets = xds_from_zarr("stack.zarr")                         # one per input line
chan_freq = xds_from_zarr("stack.zarr::SPECTRAL_WINDOW")[0].CHAN_FREQ.values
sums = {}
for ds in datasets:
    f = np.round(chan_freq[int(ds.DATA_DESC_ID[0])])           # its rest-frame channels
    w = ds.WEIGHT.values[:, None, :] * ~ds.FLAG.values
    for col in ("DATA", "MODEL_DATA", "CORRECTED_DATA"):
        num, den = sums.setdefault(col, ({}, {}))
        re = (w * ds[col].values.real).sum(axis=(0, 2))
        for fk, a, b in zip(f, re, w.sum(axis=(0, 2))):
            num[fk] = num.get(fk, 0.0) + a
            den[fk] = den.get(fk, 0.0) + b
freq = np.array(sorted(sums["DATA"][0]))
vel = 299792.458 * (line.rest_freq_hz - freq) / line.rest_freq_hz
plt.figure(figsize=(8, 3.5))
for col, c in (("DATA", "0.6"), ("MODEL_DATA", "C0"), ("CORRECTED_DATA", "k")):
    num, den = sums[col]
    plt.step(vel, [num[k] / den[k] * 1e3 for k in freq], where="mid", color=c, label=col)
plt.axvspan(*line.v_window_kms, color="C1", alpha=0.1, label="line window")
plt.axhline(0, color="0.5", lw=0.5)
plt.xlabel("velocity [km/s]"); plt.ylabel("mean Re V [mJy]"); plt.legend(fontsize=8)
plt.show()

## 3. Measuring the line

`measure_line` fits **all channels and all annuli at once** with

    Re V(channel, b) = F · φ(v; v₀, FWHM) · g(b; θ)

φ a Gaussian line profile, g the visibility of a circular Gaussian source of
FWHM θ (1 for a point source). The noise of every cell is measured from the
scatter of its visibilities (not from the weights), and the correlation
between channels (Hanning smoothing, continuum subtraction) from the
imaginary part; both enter every error.

* `mode="detect"`: centre and width fixed (known from another line, an
  optical redshift...): a matched filter, the most sensitive measurement.
* `mode="search"` (default): centre and width free; the significance comes
  from running the same search on noise (`n_null` realisations), so it
  accounts for having looked at many widths and centres.
* `size="auto"` (default): a point source and a Gaussian are both fitted and
  averaged with their Akaike weights; the error includes the difference.

Each run writes `<prefix>_results.json` (every number), `_profile.txt`,
`_uvamp.txt`, `_noise.txt`, `_diagnostics.npz`.

In [ ]:
weighting = WeightingConfig(scheme="natural", redshift_rescaling=True, primary_beam=True)
boot = BootstrapConfig(n_realisations=200, seed=1)


def show(name, r):
    s = r.summary
    ln, det, win = s["line"], s["detection"], s["window"]
    sig = det["significance_sigma"]
    print(f"{name:34s} F = {ln['flux']:.3f} ± {ln['flux_error']:.3f}  (true {TRUE_F:.3f})   "
          f"size {ln['theta_fwhm_arcsec']:.2f}\"   S/N {det['snr']:4.1f}"
          + (f" ({sig:.1f}σ)" if np.isfinite(sig) else "")
          + f"   window {win['flux']:.3f} ± {win['flux_error']:.3f}")


det = measure_line("line_stats.npy", LIST, line, weighting,
                   LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=250.0,
                                 n_null=500, n_null_phase=100),
                   boot, output_prefix="detect")
print()
show("detect (FWHM known)", det)

The three figures: the **profile** (flux density at zero spacing per channel,
i.e. the total flux density, with the fit, the ±1σ noise, the imaginary part
as a null test, residuals), the **line flux per annulus** against baseline
(flat for a point source, falling for a resolved one; the value at zero is the
total flux), and the **noise** diagnostics (noise from the scatter vs the
weights, standardised cells against N(0,1), the null distribution of the
detection S/N).

In [ ]:
from vista.extract.plots_line import plot_line_result
show_png(plot_line_result(det, line, "detect", unit="Jy", rebin=1))

In [ ]:
s = det.summary
print("line:", json.dumps({k: s["line"][k] for k in ("flux", "flux_error", "flux_error_thermal",
      "flux_error_bootstrap", "peak", "theta_fwhm_arcsec", "theta_error_arcsec", "size_model")}, indent=1))
print("size test:", json.dumps(s["size_test"], indent=1))
print("noise:", json.dumps({k: s["noise"][k] for k in ("kappa_scatter_over_weights",
      "channel_correlation", "imaginary_rms_over_noise", "line_free_rms_over_noise")}, indent=1))

What to look at in the numbers:

* `kappa_scatter_over_weights` ≈ 1.3: the weights misjudge the noise by the
  factor we put in; the scatter is used, so the errors are right anyway.
* `channel_correlation` ≈ 0.7 at lag 1: Hanning smoothing. Ignoring it would
  make the errors too small (§4).
* `imaginary_rms_over_noise` and `line_free_rms_over_noise` ≈ 1: the noise
  model describes the data. Well above 1 means sources not centred or
  asymmetric (imaginary) or residual continuum (line-free channels).
* `size_test.correlation_factor`: how much the channel correlation inflates
  the χ² difference between point and Gaussian (it is divided out).
* `flux_error_bootstrap` vs `flux_error_thermal`: a bootstrap much larger
  than thermal means the galaxies differ from one another more than the noise.

Now the **search**, with free centre and width:

In [ ]:
search = measure_line("line_stats.npy", LIST, line, weighting,
                      LineFitConfig(mode="search", n_null=500, n_null_phase=100),
                      boot, output_prefix="search")
print()
show("search (centre, width free)", search)
ln = search.summary["line"]
print(f"v0 = {ln['v0_kms']:+.0f} ± {ln['v0_error_kms']:.0f} km/s, "
      f"FWHM = {ln['fwhm_kms']:.0f} ± {ln['fwhm_error_kms']:.0f} km/s (true {SIM['fwhm_kms']:.0f})")
show_png(plot_line_result(search, line, "search", unit="Jy", rebin=2)[:1])

## 4. What each option changes

The statistics are a few MB: every variant below takes seconds. Each line
changes **one** thing with respect to the `detect` run above.

| Option | What it does | When |
|---|---|---|
| `size="point"` | point source only | the source is unresolved (known from elsewhere) |
| `size="gauss"` | Gaussian only | clearly resolved |
| `theta_prior_arcsec=(θ, σ_dex)` | soft prior on the size | a weak resolved line, with a size from the continuum or another line |
| `theta_fixed_arcsec=θ` | size fixed | the same, a hard version |
| `noise="weights"` | noise from the weights, not the scatter | weights known to be absolute |
| `channel_correlation=False` | channels treated as independent | never on Hanning-smoothed data (shown to see the effect) |
| `window_kms=(lo, hi)` | the window of the model-independent flux | |
| `WeightingConfig(scheme="democratic")` | every galaxy counts the same | the average galaxy rather than the deepest data |
| `redshift_rescaling=False` | no transport to z_ref | narrow redshift range |
| `primary_beam=False` | no primary-beam correction | data already corrected |
| `z_ref=...` | fix the reference redshift | comparing different stacks |

In [ ]:
def variant(name, weighting=weighting, **fit):
    cfg = dict(mode="detect", v0_kms=0.0, fwhm_kms=250.0, n_null=200, n_null_phase=0)
    cfg.update(fit)
    r = measure_line("line_stats.npy", LIST, line, weighting, LineFitConfig(**cfg),
                     BootstrapConfig(n_realisations=100, seed=1), verbose=False)
    show(name, r)
    return r


variant("reference (size auto)")
variant("size = point", size="point")
variant("size = gauss", size="gauss")
variant("size prior 0.8\" ± 0.15 dex", theta_prior_arcsec=(0.8, 0.15))
variant("size fixed 0.8\"", theta_fixed_arcsec=0.8)
variant("noise from the weights", noise="weights")
variant("no channel correlation", channel_correlation=False)
variant("window ±250 km/s", window_kms=(-250, 250))
variant("democratic weighting", weighting=WeightingConfig(scheme="democratic"))
variant("no redshift rescaling", weighting=WeightingConfig(redshift_rescaling=False))
variant("no primary-beam correction", weighting=WeightingConfig(primary_beam=False))
variant("wrong FWHM (150 km/s)", fwhm_kms=150.0);

Things to notice (the numbers are those of the default simulation):

* the **point-source model** loses about half the flux of this 0.8″ source —
  any extended emission is lost the same way;
* with the **noise from the weights** (wrong by κ = 1.3) the flux error
  survives, because it is rescaled by √χ²ᵣ, but the detection S/N does not:
  about 13σ instead of 10σ;
* **without the channel correlation** the errors are 1.6 times too small
  (about 19σ): never on Hanning-smoothed data;
* **democratic** weighting gives the shallow datasets the weight of the deep
  ones: a larger error, and the mean of the galaxies rather than of the data;
* **without the redshift rescaling** the nearer (brighter) galaxies push the
  flux up by ~20%; **without the primary-beam correction** it is ~2% low;
* a **wrong width** in `detect` mode loses flux (−20% with 150 instead of
  250 km/s): when the width is not known, use `search`.

## 5. The continuum

The continuum removed during the stacking (`MODEL_DATA`) was compressed into
`cont_stats.npy`: its flux and size come from the uv profile, here with the
chain of `extract_flux` (`method="continuum"`).

In [ ]:
from vista.extract.plots import plot_all

cont = extract_flux("cont_stats.npy", LIST, line, weighting,
                    profile=ProfileConfig(method="continuum"), uvfit=UVFitConfig(model="gauss"),
                    bootstrap=boot, output_prefix="continuum", verbose=False)
cs = cont.summary
print(f"continuum: {cs['flux_total'] * 1e3:.3f} ± {cs['flux_total_error'] * 1e3:.3f} mJy "
      f"(true {TRUE_C * 1e3:.3f}), size {cs.get('theta_fwhm_arcsec', float('nan')):.2f}\"")
show_png(plot_all(cont, line, "continuum", unit="Jy"), width=600)

## 6. Redoing the continuum subtraction

`DATA` is kept in the stack, so the subtraction can be redone — e.g. after
the quick look of §2 showed a feature in the line-free channels, or to test
how much the result depends on the window or the order. Then the stack is
compressed again and measured; the parameters of every subtraction are
recorded in the stack and in the results (`contsub`).

In [ ]:
line2 = LineConfig(rest_freq_ghz=REST_GHZ, v_window_kms=(-500, 500))
subtract_continuum("stack.zarr", line2, ContinuumConfig(order=1, exclude_kms=500), verbose=False)
compress_visibilities("stack.zarr", LIST, line2, binning=binning, line_column="CORRECTED_DATA",
                      out_line="line_stats_v2.npy", out_continuum="cont_stats_v2.npy",
                      resume=False, verbose=False)
redo = measure_line("line_stats_v2.npy", LIST, line2, weighting,
                    LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=250.0, n_null=200,
                                  n_null_phase=0), boot, verbose=False)
show("order 1, window ±500", redo)
show("order 0, window ±400 (§3)", det)
rec = redo.summary["contsub"]
print("\nrecorded:", {k: rec[k] for k in ("mode", "order", "v_window_kms", "exclude_kms", "date")})

A slope fitted on fewer line-free channels adds noise to every channel: the
error grows. Order 0 is the default for this reason; `order=-1` lets the BIC
decide for each dataset.

## 7. The older chain: `extract_flux`

The chain of the first version (Sec. 5.3 of the paper): the line shape is
fitted on the spectrum averaged over the annuli, then the amplitude annulus
by annulus at fixed shape, then the uv profile with a source model; errors
from the bootstrap only. It is kept for comparison: on the same statistics it
should agree with the joint fit at this S/N; the joint fit has the smaller
and better-calibrated errors at low S/N (see `docs/extraction.md`).

In [ ]:
old = extract_flux("line_stats.npy", LIST, line, weighting,
                   profile=ProfileConfig(method="template"), uvfit=UVFitConfig(model="gauss"),
                   bootstrap=boot, output_prefix="template", verbose=False)
o = old.summary
print(f"template + Gaussian uv fit: F = {o['flux_total']:.3f} ± {o['flux_total_error']:.3f} "
      f"(true {TRUE_F:.3f}), size {o.get('theta_fwhm_arcsec', float('nan')):.2f}\"")
show("joint fit, detect", det)
show_png(plot_all(old, line, "template", unit="Jy"), width=600)

## 8. The figures in your own style

`examples/plot_line.py` redraws the three figures from the output files alone
(numpy + matplotlib; it can be copied anywhere). Colours, lines, fonts and
sizes are in the `STYLE` dictionary; every figure is a function that returns
a matplotlib figure, to be changed further or saved. From a terminal:
`python plot_line.py detect --unit Jy --rebin 2 --format pdf`.

In [ ]:
import importlib
import plot_line as pl
importlib.reload(pl)

pl.STYLE["rc"].update({"font.family": "sans-serif", "font.size": 11})
pl.STYLE["model"].update(color="tab:purple", lw=2)
pl.STYLE["data"].update(color="tab:blue")
pl.STYLE["show_textbox"] = True
pl.STYLE["milli"] = True                  # mJy on the axes
pl.apply_style()                          # or pl.apply_style("my_style.mplstyle")

d = pl.load("detect")
fig = pl.plot_profile(d, unit="Jy", rebin=2, vlim=(-700, 700))
fig.axes[0].set_title("Stacked CO(3-2), simulated sample")
fig.savefig("my_profile.pdf", bbox_inches="tight")
plt.show()
fig = pl.plot_uv(d, unit="Jy")
plt.show()

## 9. The image plane (optional: needs CASA, `INSTALL_CASA = True` in §0)

`vista.export_ms` writes the stack as a Measurement Set, which CASA images
like any other. The image shows where the emission is and whether it is
symmetric — which the uv fit assumes. For the total flux of a resolved stack
the uv fit is the more robust (see `docs/extraction.md`, *uv plane or image
plane*). On Colab the first import of CASA downloads its data tables (a few
minutes).

In [ ]:
try:
    from casatasks import tclean
    from casatools import image as iatool
    HAVE_CASA = True
except ImportError:
    HAVE_CASA = False
    print("CASA not installed: set INSTALL_CASA = True in the setup cell and run it again")

if HAVE_CASA:
    from vista import export_ms
    export_ms("stack.zarr", "stack.ms", verbose=False)
    lo, hi = (line.rest_freq_hz * (1 - v / 299792.458) / 1e9 for v in (300, -300))
    for p in glob.glob("img_line*"):
        shutil.rmtree(p, ignore_errors=True)
    tclean(vis="stack.ms", imagename="img_line", datacolumn="corrected",
           spw=f"*:{lo:.5f}~{hi:.5f}GHz", specmode="mfs", imsize=160, cell="0.1arcsec",
           weighting="natural", niter=0)
    ia = iatool(); ia.open("img_line.image"); img = ia.getchunk()[:, :, 0, 0]; ia.close()
    ext = 80 * 0.1
    plt.figure(figsize=(5, 4.2))
    plt.imshow(img.T * 1e3, origin="lower", extent=(ext, -ext, -ext, ext), cmap="inferno")
    plt.colorbar(label="mJy/beam (mean over ±300 km/s)")
    plt.xlabel("ΔRA [arcsec]"); plt.ylabel("ΔDec [arcsec]"); plt.title("dirty image of the stack")
    plt.show()

## 10. Is the error bar right? A small Monte Carlo

The same sample with new noise, `N_MC` times: the scatter of the measured
fluxes should match the quoted errors, and about 68% of the runs should be
within 1σ of the truth. A few runs cannot show a bias of a few per cent — the
mean is uncertain by scatter/√N_MC: with the default 4 runs we got
1.14 ± 0.05, on 11 other runs of a similar setup 0.92 ± 0.04, together no
bias, with quoted errors (0.16) a little larger than the scatter (0.13).
Increase `N_MC` for a real test; each run takes about half a minute.

In [ ]:
N_MC = 4
mc = []
for k in range(N_MC):
    d_mc = f"mc_{k}"
    t_mc = simulate.make_sample(d_mc, rest_freq_ghz=REST_GHZ, **{**SIM, "seed": 100 + k})
    ViSta(t_mc["input_list"], verbose=False).run(
        output=f"{d_mc}/stack.zarr", central_freq=line.rest_freq_hz, velocity_range_kms=900,
        line=line, continuum=contsub, stats_line=f"{d_mc}/line.npy", binning=binning)
    r = measure_line(f"{d_mc}/line.npy", t_mc["input_list"], line, weighting,
                     LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=250.0, n_null=0,
                                   n_null_phase=0), BootstrapConfig(n_realisations=100),
                     verbose=False)
    mc.append((r.summary["line"]["flux"], r.summary["line"]["flux_error"], t_mc["line_flux_stack"]))
    shutil.rmtree(d_mc)                          # the Measurement Sets and the stack: ~1.5 GB
    print(f"run {k}: F = {mc[-1][0]:.3f} ± {mc[-1][1]:.3f}")
F, E, T = np.array(mc).T
z = (F - T) / E
print(f"\nmean F/F_true = {np.mean(F / T):.3f} ± {np.std(F / T, ddof=1) / np.sqrt(N_MC):.3f}, "
      f"scatter {np.std(F / T, ddof=1):.3f}, median quoted error {np.median(E / T):.3f}; "
      f"within 1σ: {np.mean(abs(z) <= 1):.2f}")

## 11. On your own data

1. Write the input list: one line per dataset,
   `<ms_path> <redshift> <RA> <Dec> [FIELD_ID] [SPW_IDS] [NORM]`, with the
   position of the galaxy (hh:mm:ss.s ±dd:mm:ss.s).
2. Stack with the continuum subtraction and the compression in one pass
   (§2). Choose `v_window_kms` wide enough to contain the whole line: it
   defines the channels used for the continuum.
3. Look at the stack (§2) and at `line_free_rms_over_noise`; if needed, redo
   the subtraction (§6).
4. Measure (§3): `detect` if the width is known, otherwise `search` and look
   at `significance_sigma`; check `imaginary_rms_over_noise`, the size test
   and the bootstrap/thermal ratio.
5. Make the figures in your style (§8).

The same from the command line, with the shell driver of the repository
(the package must be importable: `PYTHONPATH` points to the repository).
It stacks again, so it takes as long as §2–3; the results go to `cli/`:

In [ ]:
cmd = (f"PYTHONPATH={REPO} bash {REPO}/run_vista.sh --input {LIST} --rest-freq {REST_GHZ} "
       f"--v-window '-400 400' --velocity-range 900 --contsub before "
       f"--method joint --mode detect --fwhm 250 --unit Jy --out-dir cli")
print(cmd, "\n")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=WORK)
log = r.stdout + r.stderr
print("\n".join(l for l in log.splitlines() if l.startswith("==") or "Line:" in l)
      if r.returncode == 0 else log[-3000:])
print("\n`./run_vista.sh --help` lists every option; README.md and docs/extraction.md explain them.")

In [ ]:
# the files written in this run
for f in sorted(os.listdir(WORK)):
    if not os.path.isdir(f):
        print(f"{f:40s} {os.path.getsize(f) / 1e3:10.1f} kB")